# Building a web search function tool using DuckDuckGo, executing multi-turn tool calling, and rendering it within Gradio.


In [ ]:
!pip install -q ddgs
!pip install -q openai

In [ ]:
from ddgs import DDGS
from openai import OpenAI
from google.colab import userdata
from gradio import ChatInterface
import json
import os
import gradio as gr

In [ ]:
api_key=userdata.get('api')

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [ ]:
def search_web(query):
  results = DDGS().text(query)
  return results

In [ ]:
tools = [
    {
        "type": "function",
        "function": {  # <--- You need to nest it inside this "function" key
            "name": "search_web",  # Note: you might also want to fix the typo here to "search_web"
            "description": "Get response according to the user's query by searching from web",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A search query string",
                    },
                },
                "required": ["query"],
            },
        },
    },
]

In [ ]:
def chatbot():
  response = client.chat.completions.create(
    model = "openai/gpt-oss-20b:free",
    messages=[{
        "role": "user",
        "content": "what is the news about japanese prime minister and india"
    }],
    tools=tools,
    max_tokens=300
)
print(response.choices[0].message)

In [ ]:
if (response.choices[0].message.tool_calls):
  for call in response.choices[0].message.tool_calls:
    print('The call is: ', call.function.name)
    if call.function.name == 'search_web':
      answer = search_web(json.loads(call.function.arguments)['query'])
      print('The answer is: ', answer)

In [ ]:
gr.ChatInterface(fn=chatbot).launch(debug=True)

Updated Code

In [ ]:
import json

def chat(message, history):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful agent who is capable and aware "
                "that when to use tools and when to answer them "
                "without using tools. Observe the user's input "
                "carefully and think before replying."
            ),
        }
    ]

    # Add previous conversation history
    messages.extend(
        {
            "role": msg["role"],
            "content": (
                msg["content"][0]["text"]
                if isinstance(msg["content"], list)
                else msg["content"]
            ),
        }
        for msg in history
    )

    # Add current user message
    messages.append(
        {
            "role": "user",
            "content": message
        }
    )

    # First API call
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b:free",
        messages=messages,
        tools=tools,
    )

    assistant_message = response.choices[0].message

    # If no tool is needed, return the response
    if not assistant_message.tool_calls:
        return assistant_message.content

    # Add assistant message containing tool calls
    messages.append(assistant_message.model_dump())

    # Execute tool(s)
    for call in assistant_message.tool_calls:
        print("Tool:", call.function.name)

        if call.function.name == "search_web":
            args = json.loads(call.function.arguments)
            result = search_web(args["query"])
        else:
            result = f"Unknown tool requested: {call.function.name}"

        messages.append(
            {
                "role": "tool",
                "tool_call_id": call.id,
                "content": (
                    result
                    if isinstance(result, str)
                    else json.dumps(result)
                ),
            }
        )

    # Second API call after tool execution
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b:free",
        messages=messages,
        tools=tools,
    )

    return response.choices[0].message.content

In [ ]:
demo = gr.ChatInterface(fn=chat).launch(debug=True)